# Phase 4: does EfficientNet work with PaDiM and PatchCore?

**CRISP-DM stage:** Modelling

A compatibility test on a few photos, with EfficientNet-B0. It answers only "does it run?", not how well it finds defects.

**Runs on:** Kaggle with the GPU and internet on, to install anomalib and fetch the pretrained network. On the laptop it only checks the code, with an untrained network at a tiny size.

**Which photos:** 8 good Sheet Metal training photos. No test photos are read, so half A and half B are not involved. Scores on training photos prove only that the model runs.

**If something fails, there is no workaround.** The exact error is saved, because a failure is a finding for the report.

In [ ]:
!pip install anomalib==2.7.0

## 1. Settings

- **`efficientnet_b0`:** the smallest version, chosen to show most clearly what a very efficient network can do.
- **Layers:** matched to the ResNet layers by how much they shrink the photo: `blocks.1`, `blocks.2` and `blocks.4` shrink it by 4, 8 and 16, like `layer1`, `layer2` and `layer3`.
- **PaDiM keeps 100 features:** anomalib has no default for this network. 100 matches ResNet-18, so the table size stays the same.
- **PatchCore's coreset ratio of 0.1 and 9 neighbours:** anomalib's defaults.
- **Seed 42:** as in the other single-seed configurations.
- **PaDiM's statistics on the processor:** the same route as every PaDiM configuration, because the GPU's many-at-once inverse crashes in this setup. It is not a workaround made for EfficientNet.

In [ ]:
import csv
import gc
import time
import traceback
from pathlib import Path

import torch
from anomalib.data.utils import read_image
from anomalib.models import Padim
from anomalib.models.image.padim.torch_model import PadimModel
from anomalib.models.image.patchcore.torch_model import PatchcoreModel

USING_GPU = torch.cuda.is_available()

BACKBONE = "efficientnet_b0"
PADIM_LAYERS = ["blocks.1", "blocks.2", "blocks.4"]
PADIM_FEATURES = 100
PATCHCORE_LAYERS = ["blocks.2", "blocks.4"]
CORESET_RATIO = 0.1
NEIGHBOURS = 9
SEED = 42
RESULTS_FILE = "efficientnet_check_results.csv"

if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    IMAGE_SIZE = 256
    PHOTOS = 8
    PRETRAINED = True
else:
    # The laptop only proves the code runs: tiny photos and an untrained network
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    IMAGE_SIZE = 64
    PHOTOS = 4
    PRETRAINED = False

product_folder = None
for train_folder in INPUT_FOLDER.rglob("train"):
    if train_folder.parent.name == "sheet_metal":
        product_folder = train_folder.parent
if product_folder is None:
    raise FileNotFoundError("Could not find the Sheet Metal folder")

all_paths = sorted((product_folder / "train" / "good").glob("*.png"))
step = len(all_paths) // PHOTOS
paths = all_paths[::step][:PHOTOS]

recipe = Padim.configure_pre_processor(image_size=(IMAGE_SIZE, IMAGE_SIZE)).transform
photos = []
for path in paths:
    photos.append(read_image(path, as_tensor=True))
batch = recipe(torch.stack(photos)).to(DEVICE)
print("Photos used:", len(paths), "batch shape:", list(batch.shape))

with open(RESULTS_FILE, "w", newline="") as results_file:
    csv.writer(results_file).writerow(["method", "measure", "value"])


def record(method, measure, value):
    """Save one result straight away, and print it."""
    with open(RESULTS_FILE, "a", newline="") as results_file:
        csv.writer(results_file).writerow([method, measure, value])
    print(method, measure, value, flush=True)


def wait_for_gpu():
    """Wait for the GPU to finish, so the timer does not stop early."""
    if USING_GPU:
        torch.cuda.synchronize()


def clear_memory():
    """Free memory, so one method's peak cannot affect the other's."""
    gc.collect()
    if USING_GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def record_peak(method, measure):
    """Record the most GPU memory used since the last reset."""
    if USING_GPU:
        record(method, measure, round(torch.cuda.max_memory_allocated() / 1e9, 3))

## 2. The two tests

Each test records every step it finishes, so a failure shows exactly where it happened.

**PaDiM:** collect features on the GPU, then fit on the processor, then score on the GPU. The largest error checks that a table times its inverse gives the identity table.

**PatchCore:** collect features, select the coreset, then score. Training mode stores features, and `subsample_embedding` keeps 10 percent.

**If a step fails,** `except Exception` saves the error's type, message and last lines, then the other method still runs. It does not retry or change anything.

In [ ]:
def test_padim():
    method = "padim"
    torch.manual_seed(SEED)
    model = PadimModel(backbone=BACKBONE, layers=PADIM_LAYERS, pre_trained=PRETRAINED, n_features=PADIM_FEATURES).to(DEVICE)
    record(method, "step_built_model", "worked")
    record(method, "features_available", model.n_features_original)
    model.train()
    model.feature_extractor.eval()

    started = time.perf_counter()
    embeddings = model(batch)
    wait_for_gpu()
    record(method, "step_features", "worked")
    record(method, "seconds_per_photo_features", round((time.perf_counter() - started) / len(batch), 4))
    record(method, "embedding_shape", list(embeddings.shape))
    record_peak(method, "peak_gpu_gb_features")

    # Same route as every PaDiM configuration: statistics on the processor
    embeddings = torch.vstack(model.memory_bank).cpu()
    model.memory_bank = []
    started = time.perf_counter()
    model.gaussian.fit(embeddings)
    record(method, "step_fit_on_processor", "worked")
    record(method, "seconds_fit", round(time.perf_counter() - started, 3))

    photo_count, features, height, width = embeddings.shape
    flat = embeddings.view(photo_count, features, height * width)
    table = model.gaussian._cov(flat[:, :, 0], rowvar=False) + (0.01 + 0.00001) * torch.eye(features)
    error = float((table @ model.gaussian.inv_covariance[0] - torch.eye(features)).abs().max())
    record(method, "largest_inverse_error_first_position", round(error, 6))

    model.gaussian.to(DEVICE)
    model.eval()
    with torch.no_grad():
        output = model(batch)
    wait_for_gpu()
    record(method, "step_scoring", "worked")
    record(method, "anomaly_map_shape", list(output.anomaly_map.shape))
    record(method, "scores_all_finite", bool(torch.isfinite(output.anomaly_map).all()))
    record_peak(method, "peak_gpu_gb_scoring")


def test_patchcore():
    method = "patchcore"
    torch.manual_seed(SEED)
    model = PatchcoreModel(backbone=BACKBONE, pre_trained=PRETRAINED, layers=PATCHCORE_LAYERS, num_neighbors=NEIGHBOURS).to(DEVICE)
    record(method, "step_built_model", "worked")
    model.train()
    model.feature_extractor.eval()

    started = time.perf_counter()
    embedding = model(batch)
    wait_for_gpu()
    record(method, "step_features", "worked")
    record(method, "seconds_per_photo_features", round((time.perf_counter() - started) / len(batch), 4))
    record(method, "patches_per_photo", embedding.shape[0] // len(batch))
    record(method, "feature_length", embedding.shape[1])
    record_peak(method, "peak_gpu_gb_features")

    started = time.perf_counter()
    model.subsample_embedding(CORESET_RATIO)
    wait_for_gpu()
    record(method, "step_coreset", "worked")
    record(method, "seconds_coreset", round(time.perf_counter() - started, 3))
    record(method, "memory_bank_shape", list(model.memory_bank.shape))

    model.eval()
    with torch.no_grad():
        output = model(batch)
    wait_for_gpu()
    record(method, "step_scoring", "worked")
    record(method, "anomaly_map_shape", list(output.anomaly_map.shape))
    record(method, "scores_all_finite", bool(torch.isfinite(output.anomaly_map).all()))
    record_peak(method, "peak_gpu_gb_scoring")


for method_name, test_function in [("padim", test_padim), ("patchcore", test_patchcore)]:
    clear_memory()
    try:
        test_function()
        record(method_name, "outcome", "worked")
    except Exception as error:
        # A failure is a finding, so the exact error is saved and nothing is retried
        record(method_name, "outcome", "failed")
        record(method_name, "error_type", type(error).__name__)
        record(method_name, "error_message", str(error)[:500])
        record(method_name, "error_last_lines", traceback.format_exc()[-800:])
print("Finished")